# Assignment 25 — Prompting & LangChain Chains

This notebook follows Tasks 1–10 from the supplied assignment brief.

**Important:** Run the cells yourself. Model/API outputs are not fabricated.


In [ ]:
import os
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))
print("OPENAI_MODEL:", os.getenv("OPENAI_MODEL", "gpt-4o-mini"))

## Task 1 — PromptTemplate

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "You are a helpful assistant.\nAnswer this question simply:\n{question}"
)
print(prompt.input_variables)
print(prompt.format(question="What is LangChain?"))

## Task 2 — ChatPromptTemplate & Message Templates

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor."),
    ("human", "Explain {topic} for a beginner.")
])
for msg in chat_prompt.format_messages(topic="prompt templates"):
    print(msg.type, ":", msg.content)

## Task 3 — Pydantic Output Schema

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class Answer(BaseModel):
    answer: str
    confidence: float = Field(ge=0, le=1)
    source: str

parser = PydanticOutputParser(pydantic_object=Answer)
print(parser.get_format_instructions())

result = Answer(answer="LangChain connects components.", confidence=0.95, source="knowledge.txt")
print(result)

## Task 4 — Validation & Error Handling

In [ ]:
from pydantic import ValidationError

try:
    Answer(answer="bad confidence", confidence=1.5, source="demo")
except ValidationError as e:
    print(e)

## Task 5 — Simple Chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"), temperature=0)
simple_chain = prompt | llm | StrOutputParser()

if os.getenv("OPENAI_API_KEY"):
    print(simple_chain.invoke({"question": "What is a simple LangChain chain?"}))
else:
    print("Set OPENAI_API_KEY to run the real model call.")

## Task 6 — Conditional Chain

In [ ]:
from langchain_core.runnables import RunnableBranch

answer_prompt = ChatPromptTemplate.from_template(
    "Answer this factual question briefly: {question}"
)
fallback_prompt = ChatPromptTemplate.from_template(
    "Say that this question needs human review: {question}"
)

conditional = RunnableBranch(
    (lambda x: "?" in x["question"], answer_prompt | llm | StrOutputParser()),
    fallback_prompt | llm | StrOutputParser()
)

if os.getenv("OPENAI_API_KEY"):
    print(conditional.invoke({"question": "What is a conditional chain?"}))
else:
    print("Set OPENAI_API_KEY to execute the model branch.")

## Task 7 — Parallel Chain

In [ ]:
from langchain_core.runnables import RunnableParallel

definition_chain = (
    ChatPromptTemplate.from_template("Define {topic} in one sentence.")
    | llm | StrOutputParser()
)
summary_chain = (
    ChatPromptTemplate.from_template("Explain why {topic} is useful in one sentence.")
    | llm | StrOutputParser()
)
parallel = RunnableParallel(
    definition=definition_chain,
    summary=summary_chain
)

if os.getenv("OPENAI_API_KEY"):
    print(parallel.invoke({"topic": "LangChain"}))
else:
    print("Set OPENAI_API_KEY to execute the parallel model calls.")

## Task 8 — Runnables & LCEL

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

clean = RunnableLambda(lambda x: x.strip())
chain = RunnablePassthrough() | clean
print(chain.invoke("   LCEL example   "))

## Task 9 — LCEL-Based RAG Chain

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

docs = TextLoader("data/knowledge.txt", encoding="utf-8").load()
chunks = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=80
).split_documents(docs)

if os.getenv("OPENAI_API_KEY"):
    store = FAISS.from_documents(chunks, OpenAIEmbeddings())
    retriever = store.as_retriever(search_kwargs={"k": 3})

    def format_docs(items):
        return "\n\n".join(x.page_content for x in items)

    rag_prompt = ChatPromptTemplate.from_template(
        "Use only this context to answer.\n\nContext:\n{context}\n\nQuestion: {question}"
    )

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | rag_prompt | llm | StrOutputParser()
    )
    print(rag_chain.invoke("Why is chunk overlap important?"))
else:
    print("Set OPENAI_API_KEY to run OpenAI embeddings and the RAG chain.")

## Task 10 — Observations & Insights

1. Structured output is important because applications can validate fields before using them.
2. Modular chains make individual steps easier to replace and test.
3. Parallel chains are useful when several outputs are independent.
4. Conditional chains are useful when inputs need different paths.
5. RAG is useful when answers need to be grounded in a document collection.
6. LCEL makes these workflows easier to compose with reusable runnables.

### My actual observations
After running the notebook, replace this section with your own observations, including any errors and how you fixed them.
